In [ ]:
# Calculate mean and basin-wide SWE for each basin and ASO date

This is computed and saved in plot_spatial_comparison.py, called in spatial_comps.ipynb

In [ ]:
# Read in from the saved files
import pandas as pd
from pathlib import PurePath
import sys
sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/ucrb-isnobal/scripts/')
import processing as proc

In [ ]:
outdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/figures/spatial/mean_plots'
csv_list = h.fn_list(outdir, '*SWE*.csv')
_ = [print(PurePath(f).stem) for f in csv_list]

In [ ]:
df_list = [pd.read_csv(csv) for csv in csv_list]

In [ ]:
# Append basin and date info to each dataframe as a column
for df, csv in zip(df_list, csv_list):
    basinname = PurePath(csv).stem.split('_')[0]
    dt = PurePath(csv).stem.split('_SWE')[0][-8:]
    df['basin'] = basinname
    df['date'] = pd.to_datetime(dt, format='%Y%m%d')
df.head()

In [ ]:
# Combine into one dataframe
big_df = pd.concat(df_list)
big_df

In [ ]:
# Section out all the April rows
april_df = big_df[big_df['date'].dt.month == 4]
# Sum the total volumes for each basin and date
april_sum = april_df.groupby(['basin', 'date']).sum().reset_index()
for col in april_sum.columns:
    if 'total_volume' in col and 'Baseline' not in col:
        # Iterate through the rows
        for index, row in april_sum.iterrows():
            # rowbasin = row['basin']
            # rowdate = row['date'].strftime('%Y-%m-%d')
            # print(f"Basin: {rowbasin}, Date: {rowdate}")
            # Select the column for this particular row
            # print(f"{col}: {row[col] / 1e6:.0f} million cubic meters")
            print(f"{row[col] / 1e6:.0f}")
        # print('---')

In [ ]:
# Now do all the late dates (not April)
# Section out all the April rows
late_df = big_df[big_df['date'].dt.month != 4]
# Sum the total volumes for each basin and date
late_sum = late_df.groupby(['basin', 'date']).sum().reset_index()
for col in late_sum.columns:
    if 'total_volume' in col and 'Baseline' not in col:
        # Iterate through the rows
        for index, row in late_sum.iterrows():
            print(f"{row[col] / 1e6:.0f}")
            # print(f"{col}: {row[col] / 1e6:.0f} million cubic meters")
        # print('---')

# ARCHIVED

In [ ]:
import plot_spatial_comparison as psc


In [ ]:
def process_swe(ds, res, basin, dt, runtype):
    # omit zero value pixels for calculations
    ds_calc = ds.where(ds > 0)  # Omit zero value pixels
    # Compute basin-averaged SWE
    average_swe = ds_calc.mean().values  # in meters

    # compute the total SWE volume for the basin this is the sum of all of the swe depths multiplied by the area of each pixel depth
    total_swe = ds_calc.sum().values * res**2  # in m^3
    total_swe_taf = total_swe * 0.000810713  # convert from cubic meters to acre-feet

    # Make a dataframe of the results
    results = {
        'basin': basin,
        'date': dt,
        'runtype': runtype,
        'average_swe': average_swe,
        'total_swe_m3': total_swe,
        'total_swe_taf': total_swe_taf
    }
    df = pd.DataFrame([results])
    return df

In [ ]:
# Data extract directory
data_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/data_extracts'
basin_names = ['blue', 'animas', 'yampa']
WYs = [2021, 2022, 2023, 2024]

# Filter by ASO availability for each basin
state = 'CO'
aso_var, aso_res = 'swe', 50
isnobal_var, isnobal_res = 'specific_mass', 100
NWM_var, nwm_res = 'SNEQV', 1000
UA_var, ua_res = 'SWE', 800
mdx = 0
for basin in basin_names:
    basinname = basin.capitalize()
    for WY in WYs:
        aso_list = []
        aso_depth_fns = h.fn_list(aso_dir, f'{state}/*{basinname}*{WY}*{aso_var}*tif')
        if len(aso_depth_fns) > 0:
            # Modify the depth fns based on allowed WYs
            aso_depth_fns = [fn for fn in aso_depth_fns if any(f'{wy}' in fn for wy in WYs)]
            aso_list.append(aso_depth_fns)

            # Get the date_list
            date_list = retrieve_dates(aso_depth_fns, f'_{aso_var}')
            print(basin, date_list)

            # Get the corresponding datasets for these dates
            for jdx, dt in enumerate(date_list):
                # print(dt)
                # convert to yyyymmdd
                dt_str = dt.replace('-', '')
                isnobal_fn = h.fn_list(workdir, f'{basin}*/wy{WY}/*solar_albedo/run{dt_str}/snow.nc')[0]

                # Load iSnobal and ASO data
                print('Loading datasets for', dt)
                isnobal_ds = np.squeeze(xr.open_dataset(isnobal_fn, drop_variables=['thickness',  'snow_density',
                                                                         'liquid_water', 'temp_surf', 'temp_lower',
                                                                         'temp_snowcover', 'thickness_lower', 'water_saturation', 'projection'
                                                                         ]))
                isnobal_ds[isnobal_var] = isnobal_ds[isnobal_var] / 1000  # Convert from mm to m
                aso_ds = xr.open_dataset(aso_depth_fns[jdx])
                isnobal_df = process_swe(isnobal_ds[isnobal_var], isnobal_res, basin, dt, 'HRRR-SPIReS')
                # Deal with NWM data
                poly_fn = h.fn_list(script_dir, f'*{basin}*setup/polys/*shp')[0]
                if WY < 2023:
                    nwm_da = psc.locate_nwm(dt, WY, NWM_var, proj_fn, poly_fn)
                    # Compute the mean daily SWE for the NWM dataset (there are 8 time steps)
                    nwm_da = nwm_da.mean(dim='time')
                    nwm_df = process_swe(nwm_da, nwm_res, basin, dt, 'NWM')
                # Deal with UA data
                ua_da = psc.locate_ua(dt, WY, UA_var, poly_fn)
                ua_df = process_swe(ua_da, ua_res, basin, dt, 'UA')
                aso_df = process_swe(aso_ds['band_data'], aso_res, basin, dt, 'ASO')
                if WY < 2023:
                    df_list = [isnobal_df, nwm_df, ua_df, aso_df]
                else:
                    df_list = [isnobal_df, ua_df, aso_df]
                # Combine all dataframes
                if mdx == 0:
                    big_df = pd.concat(df_list, ignore_index=True)
                else:
                    big_df = pd.concat([big_df] + df_list, ignore_index=True)
                mdx += 1

In [ ]:
# Coerce it all data columns to floats
big_df['average_swe'] = big_df['average_swe'].astype(float)
big_df['total_swe_m3'] = big_df['total_swe_m3'].astype(float)
big_df['total_swe_taf'] = big_df['total_swe_taf'].astype(float)


In [ ]:
# Save to file
output_fn = PurePath(data_dir, 'basin_swe_summary.csv')
big_df.to_csv(output_fn, index=False)
print(f'Saved basin SWE summary to {output_fn}')

In [ ]:
# Plot it up
fig, ax = plt.subplots(figsize=(10, 6))
big_df.groupby(['basin', 'runtype'])['total_swe_taf'].plot(ax=ax, marker='o', linewidth=0)
ax.set_title('Basin-Wide SWE Summary')
ax.set_ylabel('Total SWE (TAF)')
plt.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(20, 3))
big_df.groupby(['basin', 'date', 'runtype'])['total_swe_taf'].mean().unstack().plot(kind='bar', ax=ax, rot=90)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid('lightgray', linestyle='--', linewidth=0.5)
ax.set_ylabel('Total SWE (TAF)')

In [ ]:
# The basin-average SWE for cells with non-zero SWE
fig, ax = plt.subplots(figsize=(20, 3))
big_df.groupby(['basin', 'date', 'runtype'])['average_swe'].mean().unstack().plot(kind='bar', ax=ax, rot=90)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid('lightgray', linestyle='--', linewidth=0.5)
ax.set_ylabel('Basin-average SWE (m)')

In [ ]:
# diff_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO/diffs'
# runtype = 'ua'
# dt = '20240411'
# test_fns = h.fn_list(diff_dir, f'{basin}_wy{WY}_{runtype}_*{dt}*original.nc')
# ds = xr.open_mfdataset(test_fns, combine='by_coords', drop_variables=['spatial_ref'], parallel=True)
# ds

In [ ]:
# Do this for density

# Looks like the diffs etc. are available as .nc, not sure it'll be easier reading those in?
# Filter by ASO availability for each basin
state = 'CO'
# aso_var, aso_res = 'density', 50
# isnobal_var, isnobal_res = 'snow_density', 100
# NWM_var, nwm_res = 'SNEQV', 1000
# UA_var, ua_res = 'DENSITY', 800
# mdx = 0

# To compare density, you want to 
for basin in basin_names:
    basinname = basin.capitalize()
    for WY in WYs:
        aso_list = []
        aso_depth_fns = h.fn_list(aso_dir, f'{state}/*{basinname}*{WY}*{aso_var}*tif')
        if len(aso_depth_fns) > 0:
            # Modify the depth fns based on allowed WYs
            aso_depth_fns = [fn for fn in aso_depth_fns if any(f'{wy}' in fn for wy in WYs)]
            aso_list.append(aso_depth_fns)

            # Get the date_list
            date_list = retrieve_dates(aso_depth_fns, f'_{aso_var}')
            print(basin, date_list)

            # Get the corresponding datasets for these dates
            for jdx, dt in enumerate(date_list):
                # print(dt)
                # convert to yyyymmdd
                dt_str = dt.replace('-', '')
                isnobal_fn = h.fn_list(workdir, f'{basin}*/wy{WY}/*solar_albedo/run{dt_str}/snow.nc')[0]

                # # Load iSnobal and ASO data
                # print('Loading datasets for', dt)
                # isnobal_ds = np.squeeze(xr.open_dataset(isnobal_fn, drop_variables=['thickness',  'specific_mass',
                #                                                          'liquid_water', 'temp_surf', 'temp_lower',
                #                                                          'temp_snowcover', 'thickness_lower', 'water_saturation', 'projection'
                #                                                          ]))
                # isnobal_ds[isnobal_var] = isnobal_ds[isnobal_var] / 1000  # Convert from mm to m
                # aso_ds = xr.open_dataset(aso_depth_fns[jdx])
                # isnobal_df = process_swe(isnobal_ds[isnobal_var], isnobal_res, basin, dt, 'HRRR-SPIReS')
                # # Deal with NWM data
                # poly_fn = h.fn_list(script_dir, f'*{basin}*setup/polys/*shp')[0]
                # if WY < 2023:
                #     nwm_da = psc.locate_nwm(dt, WY, NWM_var, proj_fn, poly_fn)
                #     # Compute the mean daily SWE for the NWM dataset (there are 8 time steps)
                #     nwm_da = nwm_da.mean(dim='time')
                #     nwm_df = process_swe(nwm_da, nwm_res, basin, dt, 'NWM')
                # # Deal with UA data
                # ua_da = psc.locate_ua(dt, WY, UA_var, poly_fn)
                # ua_df = process_swe(ua_da, ua_res, basin, dt, 'UA')
                # aso_df = process_swe(aso_ds['band_data'], aso_res, basin, dt, 'ASO')
                # if WY < 2023:
                #     df_list = [isnobal_df, nwm_df, ua_df, aso_df]
                # else:
                #     df_list = [isnobal_df, ua_df, aso_df]
                # # Combine all dataframes
                # if mdx == 0:
                #     big_df = pd.concat(df_list, ignore_index=True)
                # else:
                #     big_df = pd.concat([big_df] + df_list, ignore_index=True)
                # mdx += 1